# Семинар 8: Large Language Models

In [ ]:
%pip install bitsandbytes==0.48.2 transformers accelerate sentencepiece optimum torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 43.8 MB/s eta 0:00:00


In [ ]:
import gc
import os
import subprocess
from random import sample

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from datasets import load_dataset
from IPython.display import clear_output
from peft import (
    LoraConfig,
    PromptTuningConfig,
    PromptTuningInit,
    TaskType,
    get_peft_model,
)
from torch.optim import AdamW
from torch.utils.data import DataLoader
from torchmetrics.functional import accuracy
from tqdm import tqdm
from tqdm.auto import tqdm, trange
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
)

sns.set_theme()

os.environ["TOKENIZERS_PARALLELISM"] = "false"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Для создания семинара использовались материалы ШАД и курса Александра Шабалина.

## Closed-Source модели

Сейчас все основные SoTA-решения являются Closed-Source, и доступны только через веб-интерфейс, или через API с жесткими ограничениями. Удобно, если нужно прогнать несколько вопросов - попробуйте сами.


- OpenAI API (через VPN) - [openai.com/api](https://openai.com/api/)
- Chatbot Arena (удобный способ попробовать топовые LLMки, но с очень строгими ограничениями) - [arena.ai](arena.ai)
- YandexGPT Lite/Pro (поддерживает дообучение) - [console.yandex.cloud](https://console.yandex.cloud/folders/b1g4lgsfdsvocob346tv/foundation-models/overview)
- GigaChat API (без дообучения) - [developers.sber.ru](https://developers.sber.ru/docs/ru/gigachat/api/overview)


Но в демо-режиме особо не разгонишься, и ничего не автоматизируешь. Для масштабного применения придется платить за доступ к API. Как быть, если хотим классные модели, но бесплатно?

## Open-Source модели

К счастью, тут спасают модели с открытым исходным кодом. Удобнее всего их искать на [HuggingFace](https://huggingface.co/models?pipeline_tag=text-generation&sort=trending) или на уже упомянутом [ChatBot Arena](https://arena.ai) - во вкладке LeaderBoard искать модели с открытыми лицензиями. Примеры:

- [ruGPT-3.5](https://huggingface.co/ai-forever/ruGPT-3.5-13B)
- [GigaChat-20B-A3B-instruct](https://huggingface.co/ai-sage/GigaChat-20B-A3B-instruct)
- [Meta-Llama-3-8B](https://huggingface.co/meta-llama/Meta-Llama-3-8B) (требует HF token)
- [Mistral-7B-Instruct](https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2) (требует HF token)

Будем работать со следующими моделями:

-   [Qwen3-8B](https://huggingface.co/Qwen/Qwen3-8B)
-   [OpenChat-3.5](https://huggingface.co/openchat/openchat-3.5-0106)



### Загрузка модели

Давайте попробуем подгрузить и использовать такую модель. Сразу предупреждаю - мы будем мучать GPU, беспощадно. Не забудьте поставить GPU в среде выполнения.

In [ ]:
!nvidia-smi

Sat Mar  7 12:21:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
model_name = "openchat/openchat-3.5-0106"

tokenizer = transformers.LlamaTokenizer.from_pretrained(model_name, device_map=device)
tokenizer.pad_token_id = tokenizer.eos_token_id

model = transformers.AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    offload_state_dict=True,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/179 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Можем заметить, что мы уже заняли 13 ГБ из доступных 15. Далее действовать нужно очень осторожно.

In [ ]:
!nvidia-smi

Sat Mar  7 12:36:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P0             27W /   70W |   12857MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Генерация

Вспомним, что под капотом это все еще просто генеративная модель, которая предсказывает вероятности следующих токенов. Что делать с этими вероятностями дальше - можно определить с помощью стратегии генерации.

| Стратегия | Описание | Плюсы и минусы |
| --- | --- | --- |
| Greedy Search | Выбирает слово с наивысшей вероятностью как следующее слово в последовательности. | Плюсы: Простота и скорость.<br> Минусы: Может привести к повторяющемуся и несвязному тексту. |
| Семплинг с температурой | Добавляет случайность в выбор слова. Большая температура приводит к большей случайности. | Плюсы: Позволяет исследовать и получать разнообразный результат.<br> Минусы: Высокие температуры могут привести к бессмысленным результатам. |
| Семплинг по ядру (Top-p семплинг) | Выбирает следующее слово из усеченного словаря, "ядра" слов, которые имеют суммарную вероятность, превышающую предустановленный порог (p). | Плюсы: Обеспечивает баланс между разнообразием и качеством.<br> Минусы: Настройка оптимального 'p' может быть затруднительна. |
| Beam Search | Исследует множество гипотез (последовательностей слов) на каждом шаге и сохраняет 'k' наиболее вероятных, где 'k' - ширина луча. | Плюсы: Дает более надежные результаты, чем жадный поиск.<br> Минусы: Может страдать от нехватки разнообразия и приводить к общим ответам. |
| Top-k семплинг | Случайным образом выбирает следующее слово из 'k' слов с самыми высокими вероятностями. | Плюсы: Вводит случайность, увеличивая разнообразие результатов.<br> Минусы: Случайный выбор иногда может привести к менее связному тексту. |
| Нормализация длины | Предотвращает предпочтение модели более коротких последовательностей за счет деления логарифмированных вероятностей на длину последовательности, возведенную в некоторую степень. | Плюсы: Делает более длинные и потенциально более информативные последовательности более вероятными.<br> Минусы: Настройка фактора нормализации может быть сложной. |
| Стохастический Beam Search | Вводит случайность в процесс выбора 'k' гипотез в поиске пучком. | Плюсы: Увеличивает разнообразие в сгенерированном тексте.<br> Минусы: Баланс между разнообразием и качеством может быть сложно управлять. |
| Декодирование с минимальным риском Байеса (MBR) | Выбирает гипотезу (из многих), которая минимизирует ожидаемую потерю для функции потерь. | Плюсы: Оптимизирует результат в соответствии с определенной функцией потерь.<br> Минусы: Вычислительно более сложно и требует хорошо подобранную функциию потерь. |

Референсы:
- [Документация `AutoModelForCausalLM.generate()`](https://huggingface.co/docs/transformers/v4.29.1/en/main_classes/text_generation#transformers.GenerationMixin.generate)
- [Документация `AutoTokenizer.decode()`](https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.PreTrainedTokenizer.decode)
- [Статья о стратегиях генерации на Huggingface](https://huggingface.co/docs/transformers/generation_strategies)

In [ ]:
prompt = "The first discovered solar lifeform looks like"
batch = tokenizer(prompt, return_tensors="pt", return_token_type_ids=False).to(device)
print("Input batch (encoded):", batch)

Input batch (encoded): {'input_ids': tensor([[    1,   415,   907,  8324, 13024,  1411,   674,  4674,   737]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}


Greedy Search

In [ ]:
output_tokens = model.generate(**batch, max_new_tokens=64, do_sample=False)

print("\nOutput:", tokenizer.decode(output_tokens[0].cpu()))


Output: <s> The first discovered solar lifeform looks like a tiny, translucent, and gelatinous blob. It is a single-celled organism that lives in the extreme conditions of the Earth’s upper atmosphere.

The discovery of this solar lifeform has opened up a new field of study in astrobiology, which is the search for


Семплинг с температурой

In [ ]:
prompt = "The first discovered solar lifeform looks like"
batch = tokenizer(prompt, return_tensors="pt", return_token_type_ids=False).to(device)
output_tokens = model.generate(
    **batch, max_new_tokens=64, do_sample=True, temperature=0.2
)

print("\nOutput:", tokenizer.decode(output_tokens[0].cpu()))


Output: <s> The first discovered solar lifeform looks like a tiny, floating jellyfish.

The first discovered solar lifeform looks like a tiny, floating jellyfish.

The first discovered solar lifeform looks like a tiny, floating jellyfish.

The first discovered solar lifeform looks like a tiny, floating jellyfish.




In [ ]:
output_tokens = model.generate(
    **batch, max_new_tokens=64, do_sample=True, temperature=39.9
)

print("\nOutput:", tokenizer.decode(output_tokens[0].cpu()))


Output: <s> The first discovered solar lifeform looks like any earth star but its origin tells us new facs the Earth must accommodate into itself inorder  keep earth-bound entities, organinisms and living forms happy, we see another aspect with ASTRAPAC
Courtham Hall, Belsesbury Castle & Park , High Cross Street- M5


Top-K семплинг

In [ ]:
output_tokens = model.generate(
    **batch, max_new_tokens=64, do_sample=True, temperature=0.1, top_k=10
)

print("\nOutput:", tokenizer.decode(output_tokens[0].cpu()))


Output: <s> The first discovered solar lifeform looks like a tiny, translucent, and gelatinous blob.

The first discovered solar lifeform looks like a tiny, translucent, and gelatinous blob.

Text: Harshitha Kumar

The first discovered solar lifeform, known as a "plasmoid," is


Beam Search

In [ ]:
output_tokens = model.generate(**batch, max_new_tokens=64, do_sample=False, num_beams=3)

print("\nOutput:", tokenizer.decode(output_tokens[0].cpu()))


Output: <s> The first discovered solar lifeform looks like a cross between a jellyfish and a sea anemone.

The first discovered solar lifeform looks like a cross between a jellyfish and a sea anemone.

The first discovered solar lifeform looks like a cross between a jellyfish and a sea anemone.




### Создание промпта

Изначально модели заточены на генерацию - чтобы общаться с ними в привычном режиме диалога промпт нужно отформатировать. Правильный формат обычно указан в документации модели, но для некоторых моделей его можно восстановить с помощью метода apply_chat_template

In [ ]:
messages = [
    {"role": "user", "content": "What is your favourite condiment?"},
    {
        "role": "assistant",
        "content": "Well, I'm quite partial to a good squeeze of fresh lemon juice. It adds just the right amount of zesty flavour to whatever I'm cooking up in the kitchen!",
    },
    {"role": "user", "content": "Do you have mayonnaise recipes?"},
]

In [ ]:
prompt = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

In [ ]:
prompt

"<s>GPT4 Correct User: What is your favourite condiment?<|end_of_turn|>GPT4 Correct Assistant: Well, I'm quite partial to a good squeeze of fresh lemon juice. It adds just the right amount of zesty flavour to whatever I'm cooking up in the kitchen!<|end_of_turn|>GPT4 Correct User: Do you have mayonnaise recipes?<|end_of_turn|>GPT4 Correct Assistant:"

In [ ]:
batch = tokenizer(prompt, return_tensors="pt", return_token_type_ids=False).to(device)
output_tokens = model.generate(
    **batch, max_new_tokens=256, do_sample=True, temperature=0.7
)

print("\nOutput:", tokenizer.decode(output_tokens[0].cpu()))


Output: <s><s> GPT4 Correct User: What is your favourite condiment?<|end_of_turn|> GPT4 Correct Assistant: Well, I'm quite partial to a good squeeze of fresh lemon juice. It adds just the right amount of zesty flavour to whatever I'm cooking up in the kitchen!<|end_of_turn|> GPT4 Correct User: Do you have mayonnaise recipes?<|end_of_turn|> GPT4 Correct Assistant: Certainly! Here's a simple recipe for homemade mayonnaise:

Ingredients:
- 1 large egg yolk
- 1 teaspoon Dijon mustard
- 1 tablespoon white wine vinegar
- 1 teaspoon lemon juice
- 1 cup vegetable oil (such as sunflower or canola oil)
- Salt and pepper, to taste

Instructions:
1. In a large bowl, whisk together the egg yolk, mustard, vinegar, and lemon juice until well combined.
2. Slowly drizzle in the oil while continuously whisking. Start with a few drops and then gradually increase the flow of oil. This will help the mayonnaise emulsify properly.
3. Once all the oil has been added, and the mixture has thickened, season wit

У OpenChat есть два варианта ассистента - дефолтный, который реализуется с помощью apply_chat_template, и математический - для его использования нужно использовать имена Math Correct User и Math Correct Assistant.

In [ ]:
messages = [{"role": "user", "content": "10.3 − 7988.8133="}]

prompt = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
).replace("GPT4", "Math")

print(prompt)

<s>Math Correct User: 10.3 − 7988.8133=<|end_of_turn|>Math Correct Assistant:


In [ ]:
batch = tokenizer(prompt, return_tensors="pt", return_token_type_ids=False).to(device)
output_tokens = model.generate(
    **batch, max_new_tokens=256, do_sample=True, temperature=0.7
)

print("\nOutput:", tokenizer.decode(output_tokens[0].cpu()))


Output: <s><s> Math Correct User: 10.3 − 7988.8133=<|end_of_turn|> Math Correct Assistant: 10.3 - 7988.8133 = -7988.7103<|end_of_turn|>


In [ ]:
10.3 - 7988.8133

-7978.5133

### Chain-of-Thought Reasoning

Для оптимальных промптов модели необходимо давать не только примеры ответов, но и снабжать эти примеры детально описанным процессом того, как прийти к этому результату - и при генерации требовать от модели того же.

In [ ]:
prompt = """
GPT4 Correct User:
Question: The original retail price of an appliance was 60 percent more than its wholesale cost. If the appliance was actually sold for 20 percent less than the original retail price, then it was sold for what percent more than its wholesale cost?
Answer Choices: (A) 20% (B) 28% (C) 36% (D) 40% (E) 42% <|end_of_turn|>
GPT4 Correct Assistant:
Rationale: wholesale cost = 100;\noriginal price = 100*1.6 = 160;\nactual price = 160*0.8 = 128.\nAnswer: B.
Correct Answer: B <|end_of_turn|>


GPT4 Correct User:
Question: A grocer makes a 25% profit on the selling price for each bag of flour it sells. If he sells each bag for $100 and makes $3,000 in profit, how many bags did he sell?
Answer Choices: (A) 12 (B) 16 (C) 24 (D) 30 (E) 40 <|end_of_turn|>
GPT4 Correct Assistant:
Rationale: Profit on one bag: 100*1.25= 125\nNumber of bags sold = 3000/125 = 24\nAnswer is C.
Correct Answer: C <|end_of_turn|>


GPT4 Correct User:
Question: 20 marbles were pulled out of a bag of only white marbles, painted black, and then put back in. Then, another 20 marbles were pulled out, of which 1 was black, after which they were all returned to the bag. If the percentage of black marbles pulled out the second time represents their percentage in the bag, how many marbles in total Q does the bag currently hold?
Answer Choices: (A) 40 (B) 200 (C) 380 (D) 400 (E) 3200
GPT4 Correct Assistant:
Rationale: We know that there are 20 black marbles in the bag and this number represent 1/20 th of the number of all marbles in the bag, thus there are total Q of 20*20=400 marbles.\nAnswer: D.
Correct Answer: D <|end_of_turn|>


GPT4 Correct User: Question: Janice bikes at 10 miles per hour, while Jennie bikes at 20. How long until they have collectively biked 1 mile?
Answer Choices: (A) 1 minute (B) 2 minutes (C) 3 minutes (D) 4 minutes (E) 5 minutes
GPT4 Correct Assistant:
Rationale:
""".strip()

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt", return_token_type_ids=False).to(device)
output_tokens = model.generate(
    **inputs, do_sample=True, temperature=0.9, max_new_tokens=512
)
print("\nOutput:", tokenizer.decode(output_tokens[0].cpu()))


Output: <s> GPT4 Correct User:
Question: The original retail price of an appliance was 60 percent more than its wholesale cost. If the appliance was actually sold for 20 percent less than the original retail price, then it was sold for what percent more than its wholesale cost?
Answer Choices: (A) 20% (B) 28% (C) 36% (D) 40% (E) 42% <|end_of_turn|> 
GPT4 Correct Assistant:
Rationale: wholesale cost = 100;
original price = 100*1.6 = 160;
actual price = 160*0.8 = 128.
Answer: B.
Correct Answer: B <|end_of_turn|> 


GPT4 Correct User:
Question: A grocer makes a 25% profit on the selling price for each bag of flour it sells. If he sells each bag for $100 and makes $3,000 in profit, how many bags did he sell?
Answer Choices: (A) 12 (B) 16 (C) 24 (D) 30 (E) 40 <|end_of_turn|> 
GPT4 Correct Assistant:
Rationale: Profit on one bag: 100*1.25= 125
Number of bags sold = 3000/125 = 24
Answer is C.
Correct Answer: C <|end_of_turn|> 


GPT4 Correct User:
Question: 20 marbles were pulled out of a ba

In [ ]:
del model
del tokenizer
torch.cuda.empty_cache()
gc.collect()

9730

In [ ]:
torch.cuda.empty_cache()
gc.collect()

0

## Использование квантизированных моделей

Если мы жестко ограничены в ресурсах, в частности, в памяти (как в колабе), бывает полезно использовать квантизированную модель. Например, мы работаем с моделью Qwen3-8B, но хотим уменьшить объём занимаемой оперативной памяти — тогда можем квантизировать её до 8 бит и использовать в таком виде.

In [ ]:
!nvidia-smi

Sat Mar  7 12:40:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   75C    P0             44W /   70W |     157MiB /  15360MiB |    100%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
model_name = "Qwen/Qwen3-8B"

tokenizer = AutoTokenizer.from_pretrained(model_name, device_map=device)
tokenizer.pad_token_id = tokenizer.eos_token_id

quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
)

model = transformers.AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    offload_state_dict=True,
    quantization_config=quantization_config,
)

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [ ]:
prompt = "Высшая школа экономики это"
batch = tokenizer(prompt, return_tensors="pt", return_token_type_ids=False).to(device)
print("Input batch (encoded):", batch)

output_tokens = model.generate(**batch, max_new_tokens=512, do_sample=False)

print("\nOutput:", tokenizer.decode(output_tokens[0].cpu()))

Input batch (encoded): {'input_ids': tensor([[ 45610,   2247, 132775, 131862,   1478, 138361, 130193,  67879]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}

Output: Высшая школа экономики это государственное или частное учреждение? Высшая школа экономики (ВШЭ) — это частное образовательное учреждение. Оно было основано в 1992 году и находится в Москве. ВШЭ известно своими программами в области экономики, управления, права и других социальных наук. Хотя оно не является государственным, оно получает государственные субсидии и сотрудничает с государственными структурами. ВШЭ также имеет международную репутацию и предлагает программы на английском языке. 

Если вы хотите узнать больше о конкретных программах или структуре ВШЭ, дайте знать! 

Вот итог в виде списка:
- Частное образовательное учреждение
- Основано в 1992 году
- Расположено в Москве
- Специализируется на экономике, управлении, праве и других социальных науках
- Получает госу

In [ ]:
!nvidia-smi

Sat Mar  7 13:02:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   76C    P0             47W /   70W |    9623MiB /  15360MiB |     41%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----